In [ ]:
import os
import sys
import pandas as pd
from ydata_profiling import ProfileReport
current = os.getcwd()
path_to_root = os.path.join (current, '../..')
abs_path = os.path.abspath(path_to_root)
sys.path.append(abs_path)
import config

client = config.create_minio_client()

In [ ]:
object_name = "Compute_Engine/page_1.json"

try:
    response = client.get_object(config.PROVIDERS.get("google").get("bucket"), object_name=object_name)

    df = pd.read_json(response)

    response.close()
    response.release_conn()

    print ("Success. Page loaded and converted into dataframe")
    print ("Array size: Rows = ",df.shape[0], " and Columns = ", df.shape[1])

except Exception as e:
    print ("Error: ",e)


In [ ]:
#With the commnand bellow, we open the first level key : value pairs in columns, and the first level inner dicts also open, 
#in the form key.value (ex: category.serviceDisplayName, <- This was an inner dict category :{key:value, key:value})

df_flat = pd.json_normalize(df['skus'])
# df_flat.head(3)
df_flat[['skuId', 'category.serviceDisplayName', 'category.resourceFamily', 'category.usageType', 'category.resourceGroup']].head()

In [ ]:
#Here geoTaxonomy.regions looks like this: ["value"]
df_flat[['skuId','geoTaxonomy.type', 'geoTaxonomy.regions']].head()

#We apply the explode in the geoTaxonomy.regions column and store the res in a new dataframe
df_flat2 = df_flat.explode('geoTaxonomy.regions')

#The result will be: geoTaxonomy wont be a list anymore, and all the list elements will be in a single line
# df_flat2[['skuId','geoTaxonomy.type', 'geoTaxonomy.regions']].head()
df_flat2.head(3)


In [ ]:
#We apply the same proccedure as the cell above. This time we open the list serviceRegions
df_flat2[['skuId', 'serviceRegions']].head()

df_flat3 = df_flat2.explode('serviceRegions')

df_flat3[['skuId', 'serviceRegions', 'geoTaxonomy.regions']].head()

df_flat3.head()

In [ ]:
#The only column not fully opened yet is the pricingInfo: structure -> [{key1:val1, key2:val2, key3:val3, key4 : {key:val, key:[{}]}}]
df_flat3[['skuId', 'pricingInfo']].head()

#This first explode removes the list: we have now -> {key1:val1, key2:val2, key3:val3, key4 : {key:val, key:[{}]}} 
df_flat4 = df_flat3.explode('pricingInfo')
df_flat4[['skuId', 'pricingInfo']].head()
# df_flat4.head()

In [ ]:
#We are here now: {key1:val1, key2:val2, key3:val3, key4 : {key:val, key:[{}]}} -> we can open the dict with the normalize
#A new dataframe will be created with the pricing info and then concatenated with the original dataframe

pricing1 = pd.json_normalize(df_flat4['pricingInfo'])
pricing1.head()

In [ ]:
#I now have to concatenate the 2 dataframes beeing carefull though with the indexes
pricing1.index = df_flat4.index

df_flat5 = pd.concat([df_flat4.drop(columns=['pricingInfo']), pricing1], axis=1)
df_flat5.head()

In [ ]:
df_flat6 = df_flat5.explode('pricingExpression.tieredRates')

df_flat6[['skuId', 'pricingExpression.tieredRates']].head()

In [ ]:
pricing2 = pd.json_normalize(df_flat6['pricingExpression.tieredRates'])
pricing2.head()

In [ ]:
pricing2.index = df_flat6.index

df_final_flat = pd.concat([df_flat6.drop(columns=['pricingExpression.tieredRates']), pricing2], axis=1)
pd.set_option('display.max_columns', None)
df_final_flat.head()